# DocuSense — Semantic Search, Similarity Index & RAG Assistant
### Natural Language Processing and Applications (NAT820S) — Project 2
**Namibia University of Science and Technology (NUST)**

---

**Document Collection:** BBC News Dataset (1,440 unique articles, 4,981 passages across 5 categories)
**Key Technologies:** Python 3, scikit-learn, sentence-transformers (`all-MiniLM-L6-v2`), FAISS (`IndexFlatIP`), Gradio
**Group Context:** Four group members (including one member on exchange in Denmark)

> **Academic Integrity & AI Declaration:** This notebook implements a fully documented, academically defensible NLP system comparing classical lexical retrieval (TF-IDF) with dense transformer embeddings and a citation-grounded RAG assistant.

## Stage 1: Environment Setup & Dependency Installation
Installs the required Python packages reproducibly. Works on both standard CPU runtimes and GPU-accelerated Colab environments.

In [ ]:
# 1. Install dependencies silently
!pip install -q sentence-transformers faiss-cpu gradio scikit-learn pandas numpy nltk python-dotenv

import os
import sys
import json
import re
import nltk
import numpy as np
import pandas as pd

# 2. Download required NLTK WordNet corpora quietly
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('✓ Dependencies installed and core packages imported successfully.')

## Stage 2: Dataset Ingestion & Validation
Locates the BBC News dataset files, performs schema validation, checks for duplicate articles, and confirms the actual record counts.

In [ ]:
# Verify workspace structure or handle Colab repository setup
# Auto-detect repository directory across local and Colab environments
import os
import sys
for candidate in ['.', 'nlp820sGROUPASSINMENT', 'nlp820sGROUPASSINMENT-main']:
    if os.path.exists(os.path.join(candidate, 'data', 'bbc_articles.csv')):
        if candidate != '.':
            os.chdir(candidate)
        break

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

data_dir = 'data'
if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)

chunks_path = os.path.join(data_dir, 'bbc_chunks.csv')
articles_path = os.path.join(data_dir, 'bbc_articles.csv')
embeddings_path = os.path.join(data_dir, 'bbc_embeddings.npy')

print(f'Chunks file exists: {os.path.exists(chunks_path)}')
print(f'Articles file exists: {os.path.exists(articles_path)}')
print(f'Embeddings file exists: {os.path.exists(embeddings_path)}')

# Load articles and inspect ground-truth metrics
articles_df = pd.read_csv(articles_path)
chunks_df = pd.read_csv(chunks_path)

print('\n--- BBC News Corpus Summary ---')
print(f'Total Unique Articles: {len(articles_df):,}')
print(f'Total Indexed Passages: {len(chunks_df):,}')
print(f'Missing Values in Text: {articles_df["Text"].isna().sum()}')
print('\nCategory Distribution across 1,440 articles:')
print(articles_df['Category'].value_counts())

## Stage 3: Differentiated Preprocessing & Chunking
The system enforces decoupled preprocessing:
- **Lexical Preprocessing (TF-IDF):** Lowercasing, punctuation stripping, optional lemmatization.
- **Semantic Preprocessing (Transformers):** Preserves syntax, casing, negations, and numbers required for self-attention.

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lexical_preprocess(text, lemmatize=False):
    clean = re.sub(r'\s+', ' ', text).strip().lower()
    clean = re.sub(r'[^\w\s]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    if lemmatize:
        clean = ' '.join(lemmatizer.lemmatize(w) for w in clean.split())
    return clean

def semantic_preprocess(text):
    return re.sub(r'\s+', ' ', text).strip()

# Inspect sample passage chunk
sample = chunks_df.iloc[0]
print(f'Chunk ID: {sample["chunk_id"]} (Article {sample["ArticleId"]}, {sample["Category"]})')
print('Text Preview:', sample['chunk_text'][:200], '...')

## Stage 4: Classical Baseline — TF-IDF Vectorization & Retrieval
Builds a `TfidfVectorizer` with sublinear term-frequency scaling, n-gram ranges $(1, 2)$, and document frequency bounds (`max_df=0.8, min_df=2`).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit TF-IDF on all 4,981 passage chunks
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.8,
    min_df=2,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
tfidf_matrix = vectorizer.fit_transform(chunks_df['chunk_text'])
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF Index Matrix Shape: {tfidf_matrix.shape} (passages x vocabulary)')
print(f'Total Vocabulary Size: {len(feature_names):,} n-gram features')

def search_tfidf(query, k=5):
    clean_q = lexical_preprocess(query)
    q_vec = vectorizer.transform([clean_q])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top_idx = np.argsort(scores)[::-1][:k]
    
    results = chunks_df.iloc[top_idx].copy()
    results['score'] = scores[top_idx]
    return results[['chunk_id', 'ArticleId', 'Category', 'score', 'chunk_text']]

# Test query
test_hits = search_tfidf('interest rates and inflation', k=3)
display(test_hits[['chunk_id', 'Category', 'score']])

## Stage 5: Dense Semantic Embeddings & FAISS Vector Indexing
Encodes documents using `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors) and indexes them using a FAISS `IndexFlatIP` (Inner Product on L2-normalized vectors = Cosine Similarity).

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# 1. Load Pretrained Embedding Model
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'Loading sentence embedding model: {model_name}...')
embed_model = SentenceTransformer(model_name)

# 2. Load or Compute Embeddings (Reusing precomputed vectors for instant startup)
if os.path.exists(embeddings_path):
    print(f'Loading precomputed embeddings from {embeddings_path}...')
    embeddings = np.load(embeddings_path).astype(np.float32)
else:
    print('Generating embeddings from scratch (batch_size=64)...')
    embeddings = embed_model.encode(
        chunks_df['chunk_text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.save(embeddings_path, embeddings)

print(f'Embeddings Shape: {embeddings.shape} (passages x 384 dimensions)')

# 3. Build FAISS IndexFlatIP
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print(f'Total vectors in FAISS index: {faiss_index.ntotal}')

def search_semantic(query, k=5):
    clean_q = semantic_preprocess(query)
    q_emb = embed_model.encode([clean_q], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(q_emb, k)
    
    results = chunks_df.iloc[indices[0]].copy()
    results['score'] = scores[0]
    return results[['chunk_id', 'ArticleId', 'Category', 'score', 'chunk_text']]

semantic_hits = search_semantic('cost of borrowing money', k=3)
display(semantic_hits[['chunk_id', 'Category', 'score']])

## Stage 6: Comparative Retrieval Analysis (Lexical vs. Semantic)
Demonstrates the fundamental difference between keyword frequency matching (TF-IDF) and dense latent vector proximity (Sentence Transformers).

In [ ]:
def compare_query(query, k=3):
    print('='*70)
    print(f'QUERY: "{query}"')
    print('='*70)
    
    t_hits = search_tfidf(query, k)
    s_hits = search_semantic(query, k)
    
    print('\n[TF-IDF (Lexical Baseline)]')
    for _, h in t_hits.iterrows():
        print(f"  • [{h['chunk_id']}] ({h['Category']}) score={h['score']:.4f}: {h['chunk_text'][:90]}...")
        
    print('\n[Semantic Embeddings (FAISS)]')
    for _, h in s_hits.iterrows():
        print(f"  • [{h['chunk_id']}] ({h['Category']}) score={h['score']:.4f}: {h['chunk_text'][:90]}...")

# Test 1: Conceptual Synonym Query (Semantic Wins)
compare_query('cost of borrowing money and inflation impact', 3)

# Test 2: Verbatim Proper Nouns (TF-IDF Wins)
compare_query('worldcom fraud trial bernie ebbers', 3)

## Stage 7: Unsupervised Topic Discovery & Clustering
Clusters whole articles into $k=5$ latent groups using K-Means on article embeddings, extracts descriptive keywords via mean TF-IDF, and identifies representative medoids.

In [ ]:
from sklearn.cluster import KMeans

# Compute or pool article embeddings
art_embs_path = os.path.join(data_dir, 'bbc_article_embeddings.npy')
if os.path.exists(art_embs_path):
    article_embs = np.load(art_embs_path).astype(np.float32)
else:
    print('Encoding whole articles for clustering...')
    article_embs = embed_model.encode(
        articles_df['Text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.save(art_embs_path, article_embs)

# Fit K-Means
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
articles_df['cluster'] = kmeans.fit_predict(article_embs)

# Extract top keywords per cluster using article TF-IDF
art_vec = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
art_tfidf = art_vec.fit_transform(articles_df['Text'])
art_terms = art_vec.get_feature_names_out()

print('\n--- Discovered Topics & Keywords ---')
for c in sorted(articles_df['cluster'].unique()):
    mask = (articles_df['cluster'] == c).values
    mean_scores = art_tfidf[mask].mean(axis=0).A1
    top_kw = [art_terms[i] for i in np.argsort(mean_scores)[::-1][:6]]
    dominant_cat = articles_df.loc[mask, 'Category'].mode()[0]
    purity = (articles_df.loc[mask, 'Category'] == dominant_cat).mean() * 100
    print(f'Cluster {c+1} (Dominant: {dominant_cat.title()}, Purity: {purity:.1f}%): {", ".join(top_kw)}')

## Stage 8: Query Understanding & Entity-Protected Normalization
Demonstrates the query processor protecting proper nouns (e.g. *Wenger, Howard, Ebbers*) from being corrupted into common dictionary words (*wiener, toward*).

In [ ]:
from src.query_processing import get_query_processor

qp = get_query_processor()
test_queries = [
    'wenegr arsnal contrct',
    'noward truent snoker hall',
    'cost of borrowing money',
    '   ',
    '???!!!',
]

for tq in test_queries:
    res = qp.process(tq, method='embeddings')
    print(f'Raw: {tq!r} -> Valid: {res.is_valid} | Processed: {res.processed_query!r}')

## Stage 9: Grounded RAG Assistant & Citation Validation
Demonstrates question-answering with strict evidence grounding, citation auditing, and out-of-domain abstention.

In [ ]:
from src.rag import get_rag_assistant

rag = get_rag_assistant()

# Test 1: Grounded In-Domain Question
q1 = 'What did Arsene Wenger agree to regarding his contract at Arsenal?'
res1 = rag.answer_question(q1, retrieval_method='embeddings', k=3)
print(f'QUESTION: {q1}')
print(f'ANSWER:   {res1["answer"]}')
print(f'CITATIONS AUDIT: {res1["citations"]["status"]} ({res1["citations"]["valid_citations"]})')
print('-'*70)

# Test 2: Out-of-Domain Question (Must Abstain)
q2 = 'Who won the FIFA World Cup 2022 in Qatar?'
res2 = rag.answer_question(q2, retrieval_method='embeddings', k=3)
print(f'QUESTION: {q2}')
print(f'ANSWER:   {res2["answer"]}')
print(f'ABSTAINED: {res2["is_abstention"]}')

## Stage 10: Quantitative Evaluation Benchmark (30 Queries)
Executes the full 30-query evaluation benchmark and reports Precision@1, Precision@3, Precision@5, Recall@5, MRR, and nDCG@5.

In [ ]:
from src.evaluation import get_evaluation_engine

eval_engine = get_evaluation_engine()
results = eval_engine.run_comparative_evaluation(k=5)

print('\n--- DocuSense 30-Query Retrieval Benchmark Results ---')
comp_df = pd.DataFrame(results['metric_comparison_table'])
display(comp_df)

## Stage 11: Launch the Interactive Gradio Application
Launches the full 7-tab interface directly inside Colab with `share=True` to create a public temporary demo link.

In [ ]:
# Launch Gradio interface (with public 72-hour share link for Colab)
# Option A: Launch via CLI
!python app.py --share

# Option B: Inline cell launch (uncomment if preferred):
# from app import demo
# demo.launch(share=True, inbrowser=True)


## Stage 12: Troubleshooting & Reproducibility Guide

| Issue | Cause | Fix |
| :--- | :--- | :--- |
| `ModuleNotFoundError: No module named 'src'` | Colab working directory offset | Run `%cd /content/nlp820sGROUPASSINMENT-main` before running imports. |
| `FileNotFoundError: bbc_chunks.csv` | Dataset files missing from path | Ensure the `data/` directory contains `bbc_chunks.csv`, `bbc_articles.csv`, and `bbc_embeddings.npy`. |
| Gradio Share Link Fails | Colab firewall blocking port | Run `demo.launch(share=True, inbrowser=True)`. |
| Out of Memory (OOM) | High batch size during encoding | Set `batch_size=32` in `SentenceTransformer.encode()`. |
| Gemini/OpenAI API Error | Missing API key in environment | Set `os.environ['GEMINI_API_KEY'] = 'your_key'` or rely on the default local Extractive Fallback. |
